# Práctica 5: Procesamiento del Lenguaje
### UNR - TUIA - Procesamiento de Lenguaje Natural

## Librerias

In [30]:
import json
import os
import configparser
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

## Parte 1: Transformación de los Datos (ETL)

Ejercicio 1.1. Reseñas → MongoDB (o JSON)
Objetivo: Extraer información estructurada de las reseñas en texto plano y guardarlas en
formato JSON o MongoDB.

In [38]:
# 1. Leer todos los archivos de texto de data/resenas_usuarios/

directorio_resenias= "data/resenas_usuarios/"

archivos = os.listdir(directorio_resenias)
# 2. Para cada archivo, leer su contenido y almacenarlo en una lista de reseñas.
resenias = {}
for archivo in archivos:
    with open(os.path.join(directorio_resenias, archivo), 'r') as f:
        resenias[archivo] = f.read()
# 3. Imprimir el número total de reseñas leídas.
print(f"Número total de reseñas leídas: {len(resenias)}")


Número total de reseñas leídas: 85


In [40]:
print(resenias[list(resenias.keys())[0]])

Fecha: 2024-10-25 07:45
Usuario: Patricia_Ruiz
Telefono: +54 9 11 8901-2345

La cafetera es UN SUEÑO! El molinillo integrado es lo mejor.
El café queda con un aroma increíble, mucho mejor que con café ya molido.
El temporizador es perfecto, programo la noche anterior y me despierto con el café listo.
Vale cada peso!

Puntaje: 5/5
Producto: Cafetera Negra


In [ ]:
config = configparser.ConfigParser()
config.read('config.ini')
GOOGLE_API_KEY = config['GOOGLE']['API_KEY']

MODELO = 'gemma-4-31b-it' # 'gemini-2.5-flash-lite' #  # 'gemini-2.5-flash' es más rápido pero menos potente que 'gemma-4-31b-it'
llm = ChatGoogleGenerativeAI(
    model=MODELO,
    temperature=0.0,
    thinking_level="minimal",   # Configura el nivel de pensamiento al mínimo
    api_key=GOOGLE_API_KEY
)

In [27]:
def extraer_texto(content):
    """Extrae el texto limpio de la respuesta del modelo"""
    if isinstance(content, str):
        try:
            # Intenta parsear como JSON si es string
            bloques = json.loads(content)
            if isinstance(bloques, list):
                return next((item['text'] for item in bloques if item.get('type') == 'text'), content)
        except:
            pass
    elif isinstance(content, list):
        # Si es lista directamente
        return next((item['text'] for item in content if item.get('type') == 'text'), '')
    
    return str(content)


In [31]:
prompt_extraction = ChatPromptTemplate.from_messages([
    ("system", "Eres un asistente que extrae información específica de documentos."),
    ("user", """Del siguiente documento, extrae la siguiente información en formato de JSON:

- fecha
- usuario
- telefono
- producto_nombre
- producto_id
- puntaje (1-5)
- comentario
- sentimiento (positivo/neutral/negativo)
- aspectos_positivos (precio, calidad, etc.)
- aspectos_negativos (precio, calidad, etc.)

Documento:
{document}""")
])

chain_extraction = prompt_extraction | llm

response_extraction = chain_extraction.invoke({"document": resenias[0]})

print("=== INFORMACIÓN EXTRAÍDA ===\n")
respuesta_LLM = extraer_texto(response_extraction.content)
print(respuesta_LLM)

=== INFORMACIÓN EXTRAÍDA ===

```json
{
  "fecha": "2024-10-25 07:45",
  "usuario": "Patricia_Ruiz",
  "telefono": "+54 9 11 8901-2345",
  "producto_nombre": "Cafetera Negra",
  "producto_id": null,
  "puntaje": 5,
  "comentario": "La cafetera es UN SUEÑO! El molinillo integrado es lo mejor. El café queda con un aroma increíble, mucho mejor que con café ya molido. El temporizador es perfecto, programo la noche anterior y me despierto con el café listo. Vale cada peso!",
  "sentimiento": "positivo",
  "aspectos_positivos": [
    "molinillo integrado",
    "aroma del café",
    "temporizador",
    "relación calidad-precio"
  ],
  "aspectos_negativos": []
}
```


In [32]:
# convertir a JSON para verificar que es correcto, tener en cuenta que el string puede empezar con "```json" y terminar con "```", por lo que hay que eliminar esos caracteres antes de parsear

def convertir_a_json(raw_json):
    if raw_json.startswith("```json"):
        raw_json = raw_json[len("```json"):].strip()
    if raw_json.endswith("```"):
        raw_json = raw_json[:-len("```")].strip()
    try:
        data = json.loads(raw_json)
        return data
    except json.JSONDecodeError as e:
        print("\nError al parsear JSON:", e)
        return None

In [33]:
print("\n=== JSON PARSEADO ===\n")
resenia_json = convertir_a_json(respuesta_LLM)
print(resenia_json)


=== JSON PARSEADO ===

{'fecha': '2024-10-25 07:45', 'usuario': 'Patricia_Ruiz', 'telefono': '+54 9 11 8901-2345', 'producto_nombre': 'Cafetera Negra', 'producto_id': None, 'puntaje': 5, 'comentario': 'La cafetera es UN SUEÑO! El molinillo integrado es lo mejor. El café queda con un aroma increíble, mucho mejor que con café ya molido. El temporizador es perfecto, programo la noche anterior y me despierto con el café listo. Vale cada peso!', 'sentimiento': 'positivo', 'aspectos_positivos': ['molinillo integrado', 'aroma del café', 'temporizador', 'relación calidad-precio'], 'aspectos_negativos': []}


In [41]:
# conectar a mongoDB
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client["reseñas_productos"]
collection = db["reseñas"]

In [46]:
for resenia in resenias.values():
    response_extraction = chain_extraction.invoke({"document": resenia})
    respuesta_LLM = extraer_texto(response_extraction.content)
    resenia_json = convertir_a_json(respuesta_LLM)
    if resenia_json:
        # agregar a resenia_json un campo con el nombre del archivo original
        resenia_json['id_resenia'] = next((key for key, value in resenias.items() if value == resenia), "desconocido")        
        print(resenia_json)
        collection.insert_one(resenia_json)

{'fecha': '2024-10-25 07:45', 'usuario': 'Patricia_Ruiz', 'telefono': '+54 9 11 8901-2345', 'producto_nombre': 'Cafetera Negra', 'producto_id': None, 'puntaje': 5, 'comentario': 'La cafetera es UN SUEÑO! El molinillo integrado es lo mejor. El café queda con un aroma increíble, mucho mejor que con café ya molido. El temporizador es perfecto, programo la noche anterior y me despierto con el café listo. Vale cada peso!', 'sentimiento': 'positivo', 'aspectos_positivos': ['molinillo integrado', 'aroma del café', 'temporizador', 'relación calidad-precio'], 'aspectos_negativos': [], 'id_resenia': 'resena_007.txt'}
{'fecha': '2024-10-13 16:20', 'usuario': 'Romina_Calderon', 'telefono': '+54 9 11 4567-8909', 'producto_nombre': 'ventilador de torre', 'producto_id': 'P019', 'puntaje': 4, 'comentario': 'El ventilador de torre es lindo y funciona bien. Refresca toda la habitación. El control remoto es práctico. Contenta!', 'sentimiento': 'positivo', 'aspectos_positivos': ['estética', 'funcionamient

In [47]:
# recuperar todas las reseñas guardadas en MongoDB y mostrarlas
print("\n=== RESEÑAS EN MONGODB ===")
for registro in collection.find():
    print(registro)


=== RESEÑAS EN MONGODB ===
{'_id': ObjectId('6a1619a8fb7daa06050f55f1'), 'fecha': '2024-10-25 07:45', 'usuario': 'Patricia_Ruiz', 'telefono': '+54 9 11 8901-2345', 'producto_nombre': 'Cafetera Negra', 'producto_id': None, 'puntaje': 5, 'comentario': 'La cafetera es UN SUEÑO! El molinillo integrado es lo mejor. El café queda con un aroma increíble, mucho mejor que con café ya molido. El temporizador es perfecto, programo la noche anterior y me despierto con el café listo. Vale cada peso!', 'sentimiento': 'positivo', 'aspectos_positivos': ['molinillo integrado', 'aroma del café', 'temporizador', 'relación calidad-precio'], 'aspectos_negativos': [], 'id_resenia': 'resena_007.txt'}
{'_id': ObjectId('6a1619aefb7daa06050f55f2'), 'fecha': '2024-10-13 16:20', 'usuario': 'Romina_Calderon', 'telefono': '+54 9 11 4567-8909', 'producto_nombre': 'ventilador de torre', 'producto_id': 'P019', 'puntaje': 4, 'comentario': 'El ventilador de torre es lindo y funciona bien. Refresca toda la habitación. E

In [48]:
# Contar cantidad de registros en la colección
cantidad_registros = collection.count_documents({})
print(f"\nCantidad total de reseñas en MongoDB: {cantidad_registros}")


Cantidad total de reseñas en MongoDB: 85


In [ ]:
# Borrar todas las reseñas de MongoDB (descomentar si se quiere limpiar la colección)
# collection.delete_many({})

DeleteResult({'n': 5, 'ok': 1.0}, acknowledged=True)